### Kernel: R

# Ajuste do modelo de Riscos Proporcionais de COX

In [1]:
# Configuração inicial
rm(list = ls(all = TRUE))

library(pacman)
p_load(tidyverse, survival, smcure)

seed <- 123
set.seed(seed)

## Leitura dos dados de treino e teste

In [2]:
dados_treino <- read.csv("../../../dados/dados-processados/dados_treino.csv", stringsAsFactors = FALSE)
dados_teste <- read.csv("../../../dados/dados-processados/dados_teste.csv", stringsAsFactors = FALSE)

In [3]:
# Variáveis selecionadas pelo estudo anterior

vars_paper <- c(
  "stage_paper",
  "prior_diag_trat_paper",
  "hormonioterapia_paper",
  "educ_paper",
  "quimioterapia_paper",
  "macroregiao_paper"
)

## Função para preparar a base de dados para o modelo cox e de cura
### Transforma as covariáveis selecionadas pela floresta aleatória em fatores de acordo com os subgrupos identificados pela árvore aleatória


In [4]:
prepara_base <- function(df) {
  df %>%
    select(
      id_paciente,
      tempo,
      delta_t,
      rhc_estadiamento_clinico,
      rhc_diagnostico_e_tratamentos_anteriores,
      rhc_primeiro_tratamento_recebido_no_hospital_hormonioterapia,
      rhc_primeiro_tratamento_recebido_no_hospital_quimioterapia,
      instrucao,
      macroregiao
    ) %>%
    mutate(
      id_paciente = as.numeric(id_paciente),
      tempo = as.numeric(tempo),
      delta_t = as.numeric(delta_t),
      stage_paper = case_when(
        rhc_estadiamento_clinico == "I e II" ~ "I/II",
        rhc_estadiamento_clinico == "III e IV" ~ "III/IV",
        TRUE ~ NA_character_
      ),
      prior_diag_trat_paper = case_when(
        rhc_diagnostico_e_tratamentos_anteriores == "Com diag./Com trat." ~ "Com diag./Com trat.",
        rhc_diagnostico_e_tratamentos_anteriores %in% c("Com diag./Sem trat.", "Sem diag./Sem trat.") ~ "Outros",
        TRUE ~ NA_character_
      ),
      hormonioterapia_paper = case_when(
        rhc_primeiro_tratamento_recebido_no_hospital_hormonioterapia == "Sim" ~ "Sim",
        rhc_primeiro_tratamento_recebido_no_hospital_hormonioterapia == "Não" ~ "Nao",
        TRUE ~ NA_character_
      ),
      quimioterapia_paper = case_when(
        rhc_primeiro_tratamento_recebido_no_hospital_quimioterapia == "Sim" ~ "Sim",
        rhc_primeiro_tratamento_recebido_no_hospital_quimioterapia == "Não" ~ "Nao",
        TRUE ~ NA_character_
      ),
      educ_paper = case_when(
        instrucao == 0 ~ "None",
        instrucao %in% c(1, 2) ~ "Primary/Secondary+",
        TRUE ~ NA_character_
      ),
      macroregiao_paper = case_when(
        macroregiao == 2 ~ "2",
        macroregiao %in% c(1, 3) ~ "1/3",
        TRUE ~ NA_character_
      )
    ) %>%
    mutate(
      stage_paper = factor(stage_paper, levels = c("I/II", "III/IV")),
      prior_diag_trat_paper = factor(prior_diag_trat_paper, levels = c("Com diag./Com trat.", "Outros")),
      hormonioterapia_paper = factor(hormonioterapia_paper, levels = c("Sim", "Nao")),
      educ_paper = factor(educ_paper, levels = c("None", "Primary/Secondary+")),
      quimioterapia_paper = factor(quimioterapia_paper, levels = c("Sim", "Nao")),
      macroregiao_paper = factor(macroregiao_paper, levels = c("2", "1/3"))
    ) %>%
    filter(tempo > 0, !is.na(delta_t)) %>%
    drop_na(all_of(vars_paper)) %>%
    select(id_paciente, tempo, delta_t, all_of(vars_paper))
}

dados_treino_modelo <- prepara_base(dados_treino)
dados_teste_modelo <- prepara_base(dados_teste)

In [5]:
dim(dados_treino)
dim(dados_treino_modelo)

[1] 4861   26

[1] 4861    9

In [6]:
head(dados_treino_modelo)

,id_paciente,tempo,delta_t,stage_paper,prior_diag_trat_paper,hormonioterapia_paper,educ_paper,quimioterapia_paper,macroregiao_paper
,<dbl>,<dbl>,<dbl>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>
1,1,16,1,III/IV,Outros,Nao,None,Sim,1/3
2,2,60,1,III/IV,Outros,Sim,None,Sim,1/3
3,5,69,1,III/IV,Outros,Nao,Primary/Secondary+,Sim,1/3
4,6,7,1,III/IV,Outros,Nao,Primary/Secondary+,Sim,1/3
5,7,27,1,III/IV,Outros,Sim,Primary/Secondary+,Sim,1/3
6,8,49,1,III/IV,Outros,Nao,Primary/Secondary+,Sim,1/3


## Cox simples e modelo com fração de cura

In [7]:
form_latencia <- as.formula(
  paste("Surv(tempo, delta_t) ~", paste(vars_paper, collapse = " + "))
)

modelo_cox <- coxph(form_latencia, data = dados_treino_modelo)

summary(modelo_cox)

Call:
coxph(formula = form_latencia, data = dados_treino_modelo)

  n= 4861, number of events= 575 

                                 coef exp(coef) se(coef)      z Pr(>|z|)    
stage_paperIII/IV             1.12199   3.07096  0.08988 12.483  < 2e-16 ***
prior_diag_trat_paperOutros   0.13262   1.14181  0.26236  0.505 0.613221    
hormonioterapia_paperNao      0.54554   1.72554  0.09381  5.816 6.04e-09 ***
educ_paperPrimary/Secondary+ -0.39897   0.67101  0.09009 -4.429 9.48e-06 ***
quimioterapia_paperNao       -0.37475   0.68746  0.09981 -3.755 0.000174 ***
macroregiao_paper1/3          0.64774   1.91121  0.12188  5.314 1.07e-07 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

                             exp(coef) exp(-coef) lower .95 upper .95
stage_paperIII/IV               3.0710     0.3256    2.5749    3.6625
prior_diag_trat_paperOutros     1.1418     0.8758    0.6828    1.9095
hormonioterapia_paperNao        1.7255     0.5795    1.4357    2.0738
educ_paperPr

In [8]:
# Preparação da base de dados para o modelo de cura

X_cura <- model.matrix(~ ., data = dados_treino_modelo[, vars_paper, drop = FALSE])
X_cura <- X_cura[, colnames(X_cura) != "(Intercept)", drop = FALSE]
X_cura <- X_cura[, apply(X_cura, 2, function(x) stats::sd(x, na.rm = TRUE) > 0), drop = FALSE]

colnames(X_cura) <- make.names(colnames(X_cura))

dados_treino_cura <- data.frame(
  tempo = dados_treino_modelo$tempo,
  delta_t = as.integer(dados_treino_modelo$delta_t == 1),
  X_cura,
  check.names = TRUE
)

# Fórmulas para o modelo de cura
covs_cura <- setdiff(names(dados_treino_cura), c("tempo", "delta_t"))
form_latencia_cura <- as.formula(paste("Surv(tempo, delta_t) ~", paste(covs_cura, collapse = " + ")))
form_incidencia_cura <- as.formula(paste("~", paste(covs_cura, collapse = " + ")))

In [9]:
# Base de dados com nomes adequados e categorias como fatores 0 ou 1

head(dados_treino_cura)

,tempo,delta_t,stage_paperIII.IV,prior_diag_trat_paperOutros,hormonioterapia_paperNao,educ_paperPrimary.Secondary.,quimioterapia_paperNao,macroregiao_paper1.3
,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,16,1,1,1,1,0,0,1
2,60,1,1,1,0,0,0,1
3,69,1,1,1,1,1,0,1
4,7,1,1,1,1,1,0,1
5,27,1,1,1,0,1,0,1
6,49,1,1,1,1,1,0,1


In [10]:
# Formas para o ajuste do modelo de cura

form_latencia_cura
form_incidencia_cura

Surv(tempo, delta_t) ~ stage_paperIII.IV + prior_diag_trat_paperOutros + 
    hormonioterapia_paperNao + educ_paperPrimary.Secondary. + 
    quimioterapia_paperNao + macroregiao_paper1.3

~stage_paperIII.IV + prior_diag_trat_paperOutros + hormonioterapia_paperNao + 
    educ_paperPrimary.Secondary. + quimioterapia_paperNao + macroregiao_paper1.3

In [11]:
# Modelo com fracao de cura (incidencia + latencia)

modelo_cura <- smcure(
  form_latencia_cura,
  cureform = form_incidencia_cura,
  data = dados_treino_cura,
  model = "ph",
  Var = FALSE
)

summary(modelo_cura)

Program is running..be patient... done.
Call:
smcure(formula = form_latencia_cura, cureform = form_incidencia_cura, 
    data = dados_treino_cura, model = "ph", Var = FALSE)

Cure probability model:
                               Estimate
(Intercept)                  -2.3093793
stage_paperIII.IV             1.0394421
prior_diag_trat_paperOutros  -0.1506192
hormonioterapia_paperNao      0.2039464
educ_paperPrimary.Secondary. -0.2517363
quimioterapia_paperNao       -0.5434058
macroregiao_paper1.3          0.7500786


Failure time distribution model:
                                Estimate
stage_paperIII.IV             0.54600387
prior_diag_trat_paperOutros   0.66203507
hormonioterapia_paperNao      0.86551253
educ_paperPrimary.Secondary. -0.55628939
quimioterapia_paperNao        0.26530308
macroregiao_paper1.3          0.04087434


          Length Class  Mode     
logistfit   30   glm    list     
b            7   -none- numeric  
beta         6   -none- numeric  
call         6   -none- call     
bnm          7   -none- character
betanm       6   -none- character
s         4861   -none- numeric  
Time      4861   -none- numeric  

## S(t) nos dados de teste.

In [12]:
# Grade tempos para avaliação dos modelos nos dados de teste
tempos_evento_ibs <- seq(1, 131)

In [13]:
# Previsao do modelo Cox para teste nos tempos_evento_teste
risco_individual <- predict(modelo_cox, newdata = dados_teste_modelo, type = "lp")
risco_base <- basehaz(modelo_cox, centered = FALSE)

# Risco acumulado para cada tempo de evento
H0_t <- stepfun(risco_base$time, c(0, risco_base$hazard), right = TRUE)(tempos_evento_ibs)

## Fórmula para obter S(t) para o modelo de COX Proportional Hazards

$S(t) = \exp(-H_0(t) \times \exp(X\beta))$

In [14]:
# Matriz de sobrevivência para cada paciente nos tempos de interesse
S_mat_cox <- exp(-outer(H0_t, exp(risco_individual)))

# Formato longo
pred_event_cox <- bind_rows(lapply(seq_len(nrow(dados_teste_modelo)), function(i) {
  data.frame(
    ID = dados_teste_modelo$id_paciente[i],
    TIME = tempos_evento_ibs,
    PRED_S = as.numeric(S_mat_cox[, i]),
    stringsAsFactors = FALSE
  )
}))

# Exportar previsões do modelo
arquivo_pred_cox <- "../../../ajuste-modelos/cox-cura/dados/saida/predicao_cox.csv"
write.csv(pred_event_cox, arquivo_pred_cox, row.names = FALSE)

## Pevisão do modelo de cura para os dados de teste

In [15]:
# Preparação da base dos dados de teste
X_cura_teste <- model.matrix(~ ., data = dados_teste_modelo[, vars_paper, drop = FALSE])
X_cura_teste <- X_cura_teste[, colnames(X_cura_teste) != "(Intercept)", drop = FALSE]
X_cura_teste <- as.data.frame(X_cura_teste)


colnames(X_cura_teste) <- make.names(colnames(X_cura_teste))

In [16]:
# Transformar para matriz.
X_cura_teste <- as.matrix(X_cura_teste[, covs_cura, drop = FALSE])

In [17]:
head(X_cura_teste)

,stage_paperIII.IV,prior_diag_trat_paperOutros,hormonioterapia_paperNao,educ_paperPrimary.Secondary.,quimioterapia_paperNao,macroregiao_paper1.3
1,1,1,1,0,1,1
2,1,1,1,0,0,0
3,0,1,1,1,0,1
4,1,1,0,1,0,1
5,0,1,1,1,0,0
6,1,1,0,1,0,1


In [18]:
# Predição do modelo

pred_cura_raw <- predictsmcure(
  modelo_cura,
  newX = X_cura_teste,
  newZ = X_cura_teste,
  model = "ph"
)

pred_cura_df <- as.data.frame(pred_cura_raw$prediction)

In [19]:
head(pred_cura_raw)

$call
predictsmcure(object = modelo_cura, newX = X_cura_teste, newZ = X_cura_teste, 
    model = "ph")

$newuncureprob
             1         2         3        4          5        6         7
[1,] 0.2669922 0.2285336 0.1470774 0.284507 0.07531399 0.284507 0.1232879
              8        9        10       11        12       13        14
[1,] 0.04516595 0.284507 0.1232879 0.284507 0.3277746 0.220686 0.1470774
            15        16        17       18         19       20        21
[1,] 0.1232879 0.3854376 0.1232879 0.284507 0.07550408 0.284507 0.1871941
           22        23        24        25         26        27        28
[1,] 0.284507 0.1470774 0.3277746 0.3728853 0.07531399 0.1945744 0.1232879
             29        30        31       32       33       34       35
[1,] 0.07871045 0.3277746 0.1141151 0.284507 0.284507 0.284507 0.284507
            36        37         38        39        40        41        42
[1,] 0.2974819 0.2669922 0.06228445 0.3277746 0.1531741 0.2669922 0.3

In [20]:
head(pred_cura_df)

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,⋯,V1207,V1208,V1209,V1210,V1211,V1212,V1213,V1214,V1215,Time
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,0.8409651,0.8887993,0.9697054,0.9561070,0.9850420,0.9561070,0.9885957,0.9886749,0.9561070,0.9885957,⋯,0.8923419,0.8923419,0.8923419,0.9142426,0.9405365,0.9910246,0.9886749,0.9910246,0.9850420,16
2,0.7354850,0.7787522,0.8975723,0.8351563,0.9486679,0.8351563,0.9513642,0.9649923,0.8351563,0.9513642,⋯,0.7140856,0.7140856,0.7140856,0.7363190,0.8387665,0.9637509,0.9649923,0.9637509,0.9486679,60
3,0.7341527,0.7755938,0.8896029,0.8192286,0.9445432,0.8192286,0.9454362,0.9627764,0.8192286,0.9454362,⋯,0.7020378,0.7020378,0.7020378,0.7198439,0.8315478,0.9597366,0.9627764,0.9597366,0.9445432,69
4,0.9261173,0.9515037,0.9883506,0.9834448,0.9942642,0.9834448,0.9957921,0.9955680,0.9834448,0.9957921,⋯,0.9564799,0.9564799,0.9564799,0.9664033,0.9760743,0.9966576,0.9955680,0.9966576,0.9942642,7
5,0.7879916,0.8428641,0.9512652,0.9278648,0.9758627,0.9278648,0.9807895,0.9821254,0.9278648,0.9807895,⋯,0.8358332,0.8358332,0.8358332,0.8648047,0.9088801,0.9850378,0.9821254,0.9850378,0.9758627,27
6,0.7413891,0.7893402,0.9138280,0.8654342,0.9569948,0.8654342,0.9617841,0.9698165,0.8654342,0.9617841,⋯,0.7437699,0.7437699,0.7437699,0.7723318,0.8562335,0.9710416,0.9698165,0.9710416,0.9569948,49


In [21]:
# Extração e ordenação das predições brutas
tempos_cura <- as.numeric(pred_cura_df[["Time"]])
S_cura_mat <- as.matrix(pred_cura_df[, setdiff(names(pred_cura_df), "Time"), drop = FALSE])

ordem_tempos <- order(tempos_cura)
tempos_cura_ord <- tempos_cura[ordem_tempos]
S_cura_mat_ord <- S_cura_mat[ordem_tempos, , drop = FALSE]

# Interpolação para a grade de tempos de avaliação
S_cura_interp <- sapply(seq_len(ncol(S_cura_mat_ord)), function(j) {
  suppressWarnings(
    approx(
      x = tempos_cura_ord,
      y = as.numeric(S_cura_mat_ord[, j]),
      xout = tempos_evento_ibs,
      method = "constant",
      f = 0,
      rule = 2,
      ties = min 
    )$y
  )
})

if (is.vector(S_cura_interp)) S_cura_interp <- matrix(S_cura_interp, ncol = 1)

In [22]:
# Estruturação no formato longo para as métricas de avaliação
pred_event_cura <- bind_rows(lapply(seq_len(ncol(S_cura_interp)), function(i) {
  data.frame(
    ID = dados_teste_modelo$id_paciente[i],
    TIME = tempos_evento_ibs,
    PRED_S = as.numeric(S_cura_interp[, i]),
    stringsAsFactors = FALSE
  )
}))

# Exportação dos resultados
arquivo_pred_cura <- "../../../ajuste-modelos/cox-cura/dados/saida/predicao_cura.csv"
write.csv(pred_event_cura, arquivo_pred_cura, row.names = FALSE)